# Pré-processamento textual e extração de entidades

Demonstração da entrega da issue #123 — a etapa do pipeline de PLN que antecede a classificação de intenções.

O código não vive aqui: ele está em `src/backend/app/nlp/`, importável pela API conforme o RNF10. Este notebook exercita esse módulo sobre o conjunto sintético e mede o resultado.

| Etapa | Módulo | O que faz |
| --- | --- | --- |
| Pré-processamento | `app.nlp.preprocessing` | Normaliza, tokeniza e reduz a radical, na `config_22` da Seção 8.10.3 |
| Extração de entidades | `app.nlp.entities` | Identifica projeto, subportfólio, período, indicador e demais elementos do domínio |

As duas resolvem perguntas diferentes: o classificador decide **o que** foi pedido, a extração decide **sobre o que**.

In [1]:
import sys, json, re, random, unicodedata, collections
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src' / 'backend'))

from app.nlp.preprocessing import SELECTED_CONFIG, preprocess_text, tokenize
from app.nlp.entities import extract_entities

ARTEFATOS = RAIZ / 'src' / 'notebooks' / 'artefatos'
NOTEBOOK_BENCHMARK = RAIZ / 'src' / 'notebooks' / 'classificador_intencoes_metro.ipynb'
dataset = pd.read_csv(ARTEFATOS / 'dataset_intencoes.csv')
print(f'{len(dataset)} solicitações sintéticas | {dataset.intencao.nunique()} intenções')

5353 solicitações sintéticas | 31 intenções


## 1. Pré-processamento

A `config_22` foi escolhida por validação cruzada agrupada entre 36 configurações, e está registrada na Seção 8.10.3 do `Projeto.md`. As stopwords são **preservadas**: "qual", "quando", "quanto" e "quem" distinguem tipos de solicitação.

In [2]:
print(json.dumps(dict(SELECTED_CONFIG), ensure_ascii=False, indent=2))

{
  "lowercase": true,
  "tokenizer": "regex",
  "remove_stopwords": false,
  "remove_punctuation": true,
  "morphology": "stemming",
  "stemmer_type": "snowball",
  "ngram_range": null,
  "language": "portuguese",
  "regex_pattern": "[^\\W_]+"
}


In [3]:
exemplos = [
    'Qual é o status da extensão da Linha 5-Lilás?',
    'compare o Linha 2-Verde e o PRJ-2031 por custo no acumulado do ano',
    'me avise no Teams se o SPI cair 15%',
]
for texto in exemplos:
    print(f'{texto}\n   -> {tokenize(texto)}\n')

Qual é o status da extensão da Linha 5-Lilás?
   -> ['qual', 'é', 'o', 'status', 'da', 'extensã', 'da', 'linh', '5', 'lilás']

compare o Linha 2-Verde e o PRJ-2031 por custo no acumulado do ano
   -> ['compar', 'o', 'linh', '2', 'verd', 'e', 'o', 'prj', '2031', 'por', 'cust', 'no', 'acumul', 'do', 'ano']

me avise no Teams se o SPI cair 15%
   -> ['me', 'avis', 'no', 'teams', 'se', 'o', 'spi', 'cair', '15']



### 1.1 Equivalência com o notebook do benchmark

Os vetorizadores TF-IDF serializados em `classificador_intencoes.joblib` foram ajustados sobre os tokens da função original. Se o módulo portado divergir em uma única frase, o modelo treinado deixa de casar. A verificação abaixo compara as duas implementações sobre **todo** o conjunto.

In [4]:
nb = json.loads(NOTEBOOK_BENCHMARK.read_text(encoding='utf-8'))
origem = ''.join(nb['cells'][13]['source'])
origem = origem[:origem.index('# ---- Observação')]

escopo = {}
exec(
    'from nltk.tokenize import word_tokenize, wordpunct_tokenize, WhitespaceTokenizer, RegexpTokenizer\n'
    'from nltk.corpus import stopwords\n'
    'from nltk.stem import SnowballStemmer, RSLPStemmer, WordNetLemmatizer\n'
    'from nltk.util import ngrams\n' + origem,
    escopo,
)
original = escopo['preprocess_text']

config = dict(SELECTED_CONFIG)
divergencias = sum(
    original(t, **config) != preprocess_text(t, **config) for t in dataset.texto
)
print(f'{len(dataset)} frases comparadas -> {divergencias} divergências')
assert divergencias == 0, 'o módulo portado mudou a tokenização; o modelo treinado seria invalidado'

5353 frases comparadas -> 0 divergências


## 2. Extração de entidades

A extração roda sobre o **texto cru**, antes da tokenização — o tokenizador da `config_22` quebra em `[^\W_]+` e separaria `PRJ-2031` em `PRJ` e `2031`.

Duas fontes se combinam: o léxico do domínio, em `app/nlp/data/lexico_dominio.json`, e padrões para valores fora de lista fechada (código de projeto, percentual, prazo, data).

In [5]:
def mostrar(texto):
    print(texto)
    for e in extract_entities(texto):
        marca = '  (ambígua: ' + ', '.join(e.tipos) + ')' if e.ambigua else ''
        print(f'   {e.tipo:13} {e.texto!r:38} -> {e.valor!r}{marca}')
    print()

for texto in exemplos:
    mostrar(texto)

Qual é o status da extensão da Linha 5-Lilás?
   projeto       'extensão da Linha 5-Lilás'            -> 'extensão da Linha 5-Lilás'

compare o Linha 2-Verde e o PRJ-2031 por custo no acumulado do ano
   projeto       'Linha 2-Verde'                        -> 'Linha 2-Verde'
   projeto       'PRJ-2031'                             -> 'PRJ-2031'
   criterio      'custo'                                -> 'custo'
   periodo       'no acumulado do ano'                  -> 'no acumulado do ano'

me avise no Teams se o SPI cair 15%
   canal         'no Teams'                             -> 'no Teams'
   indicador     'SPI'                                  -> 'SPI'
   percentual    '15%'                                  -> '15%'



### 2.1 Robustez às variações de superfície

40% do conjunto recebe ruído de estilo: maiúsculas, remoção de acento, cortesia no início. O casamento ignora caixa e acento, e devolve sempre o valor canônico do catálogo.

In [6]:
variantes = [
    'qual é o status da modernização da Linha 3-Vermelha?',
    'QUAL É O STATUS DA MODERNIZAÇÃO DA LINHA 3-VERMELHA?',
    'qual e o status da modernizacao da Linha 3-Vermelha?',
    'pfv qual e o status da modernizacao da Linha 3-Vermelha',
]
for texto in variantes:
    projeto = [e.valor for e in extract_entities(texto) if e.tipo == 'projeto']
    print(f'{texto[:58]:60} -> {projeto}')

qual é o status da modernização da Linha 3-Vermelha?         -> ['modernização da Linha 3-Vermelha']
QUAL É O STATUS DA MODERNIZAÇÃO DA LINHA 3-VERMELHA?         -> ['modernização da Linha 3-Vermelha']
qual e o status da modernizacao da Linha 3-Vermelha?         -> ['modernização da Linha 3-Vermelha']
pfv qual e o status da modernizacao da Linha 3-Vermelha      -> ['modernização da Linha 3-Vermelha']


### 2.2 Termos ambíguos

`aderência ao cronograma` é critério de comparação **e** indicador de desempenho. Qual dos dois vale depende da intenção — que o extrator não conhece. Em vez de arbitrar, ele registra os dois tipos candidatos e deixa a escolha para quem já classificou o pedido.

In [7]:
mostrar('compare os projetos do subportfólio de Expansão por aderência ao cronograma')
mostrar('qual a aderência ao cronograma do Linha 4-Amarela')

compare os projetos do subportfólio de Expansão por aderência ao cronograma
   subportfolio  'subportfólio de Expansão'             -> 'subportfólio de Expansão'
   indicador     'aderência ao cronograma'              -> 'aderência ao cronograma'  (ambígua: indicador, criterio)

qual a aderência ao cronograma do Linha 4-Amarela
   indicador     'aderência ao cronograma'              -> 'aderência ao cronograma'  (ambígua: indicador, criterio)
   projeto       'Linha 4-Amarela'                      -> 'Linha 4-Amarela'



## 3. Avaliação

O gabarito não é inferido do texto — seria circular. O conjunto sintético foi gerado preenchendo *slots* em templates, com semente fixa por template. Reproduzindo essa geração, sabemos exatamente **qual valor foi inserido em cada frase**.

In [8]:
def celula_executavel(fonte):
    """Remove as linhas de exibição do notebook do benchmark.

    Só interessam as definições de TEMPLATES, SLOTS e das funções de geração;
    `display(...)` e `print(...)` dependem do ambiente Jupyter do outro notebook.
    """
    linhas, pulando = [], False
    for linha in fonte.split('\n'):
        limpa = linha.strip()
        if pulando:
            pulando = not limpa.endswith(')')
            continue
        if limpa.startswith(('display(', 'print(')):
            pulando = limpa.count('(') > limpa.count(')')
            continue
        if re.match(r'^(df|_)\s*=|^df\[', linha):
            continue
        linhas.append(linha)
    return '\n'.join(linhas)

celulas = {i: ''.join(c['source']) for i, c in enumerate(nb['cells'])}
gerador = {'random': random, 're': re, 'unicodedata': unicodedata, 'pd': pd, 'SEED': 42}
for indice in (3, 5, 6, 7):
    fonte = celulas[indice]
    if indice == 7:
        fonte = fonte[:fonte.index('df = gerar_dataset(')]
    exec(celula_executavel(fonte), gerador)

TEMPLATES, SLOTS = gerador['TEMPLATES'], gerador['SLOTS']
print(f'{sum(len(v) for v in TEMPLATES.values())} templates | {len(SLOTS)} slots')

437 templates | 15 slots


In [9]:
def gerar_com_gabarito():
    """Repete a geração do conjunto, guardando os valores inseridos em cada frase."""
    linhas = []
    for intencao, tpls in TEMPLATES.items():
        for indice, template in enumerate(tpls):
            template_id = f'{intencao}#{indice:02d}'
            rng = random.Random(f"{gerador['SEED']}-{template_id}")
            campos = gerador['_campos'](template)
            vistos, tentativas = set(), 0
            while len(vistos) < gerador['N_VARIANTES'] and tentativas < gerador['N_VARIANTES'] * 40:
                tentativas += 1
                valores = {
                    campo: rng.choice(SLOTS['proj' if campo == 'proj2' else campo])
                    for campo in campos
                }
                if valores.get('proj') and valores.get('proj') == valores.get('proj2'):
                    continue
                texto = re.sub(r'\s+', ' ', template.format(**valores)).strip()
                if rng.random() < gerador['P_VARIACAO_SUPERFICIAL']:
                    variante = gerador['variacao_superficial'](texto, rng)
                    if variante != texto:
                        texto = variante
                if texto.lower() in vistos:
                    continue
                vistos.add(texto.lower())
                linhas.append({'texto': texto, 'template': template, 'valores': valores})
    return linhas

def chave(texto):
    t = re.sub(r'[^\w\s]', ' ', gerador['_sem_acento'](texto.lower()))
    return re.sub(r'\s+', ' ', t).strip()

vistos, gabarito = set(), []
for linha in gerar_com_gabarito():
    k = chave(linha['texto'])
    if k not in vistos:
        vistos.add(k)
        gabarito.append(linha)

reproduzidas = len({l['texto'] for l in gabarito} & set(dataset.texto))
print(f'{len(gabarito)} frases reconstruídas | {reproduzidas} coincidem com o conjunto versionado '
      f'({100 * reproduzidas / len(dataset):.1f}%)')

5353 frases reconstruídas | 5353 coincidem com o conjunto versionado (100.0%)


### 3.1 Precisão, revocação e F1 por tipo

Um termo escrito **no próprio template** — como "prazo" em *"qual é o prazo do {proj}?"* — é uma menção real do domínio, mas não aparece no gabarito, que só registra os valores sorteados. Contá-lo como erro puniria o extrator por acertar. Por isso ele é creditado, e a contagem estrita fica registrada ao lado.

In [10]:
SLOT2TIPO = {
    'proj': 'projeto', 'proj2': 'projeto', 'sub': 'subportfolio', 'per': 'periodo',
    'resp': 'responsavel', 'doc': 'documento', 'crit': 'criterio', 'ind': 'indicador',
    'status': 'status', 'risco': 'risco', 'freq': 'frequencia', 'canal': 'canal',
    'data': 'data', 'pct': 'percentual', 'dias': 'prazo_dias',
}
normalizar = lambda s: ''.join(unicodedata.normalize('NFD', c)[0].lower() for c in s)

tp = collections.Counter(); fn = collections.Counter()
fp_estrito = collections.Counter(); fp_real = collections.Counter()

for linha in gabarito:
    no_template = normalizar(re.sub(r'\{\w+\}', ' ', linha['template']))
    esperado = collections.Counter(
        (SLOT2TIPO[s], normalizar(v)) for s, v in linha['valores'].items() if s in SLOT2TIPO
    )
    obtido = collections.Counter()
    for e in extract_entities(linha['texto']):
        valor = normalizar(e.valor)
        valor = valor.split()[0] if e.tipo == 'prazo_dias' else valor
        for tipo in e.tipos:            # entidade ambígua conta para qualquer candidato
            obtido[(tipo, valor)] += 1
    for chave_ent in set(esperado) | set(obtido):
        tipo, valor = chave_ent
        esperados, obtidos = esperado[chave_ent], obtido[chave_ent]
        tp[tipo] += min(esperados, obtidos)
        fn[tipo] += max(esperados - obtidos, 0)
        excedente = max(obtidos - esperados, 0)
        if excedente:
            fp_estrito[tipo] += excedente
            if valor not in no_template:
                fp_real[tipo] += excedente

def metricas(T, F, N):
    p = T / (T + F) if T + F else 0.0
    r = T / (T + N) if T + N else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

linhas = []
for tipo in sorted(set(tp) | set(fn) | set(fp_estrito)):
    p, r, f1 = metricas(tp[tipo], fp_real[tipo], fn[tipo])
    p_estrita, _, _ = metricas(tp[tipo], fp_estrito[tipo], fn[tipo])
    linhas.append({'tipo': tipo, 'precisao': p, 'revocacao': r, 'f1': f1,
                   'precisao_estrita': p_estrita, 'suporte': tp[tipo] + fn[tipo]})

tabela = pd.DataFrame(linhas).set_index('tipo')
p, r, f1 = metricas(sum(tp.values()), sum(fp_real.values()), sum(fn.values()))
display(tabela.round(3))
print(f'\nGLOBAL  precisão={p:.3f}  revocação={r:.3f}  F1={f1:.3f}  '
      f'({sum(tp.values()) + sum(fn.values())} entidades em {len(gabarito)} frases)')
print(f'Erros restantes: {sum(fp_real.values())} falsos positivos, {sum(fn.values())} falsos negativos')

,precisao,revocacao,f1,precisao_estrita,suporte
tipo,,,,,
canal,1.000,1.000,1.000,1.000,121
criterio,0.875,1.000,0.933,0.187,84
data,1.000,1.000,1.000,1.000,231
documento,0.984,1.000,0.992,0.582,181
frequencia,1.000,1.000,1.000,1.000,143
indicador,0.930,1.000,0.964,0.670,120
percentual,1.000,1.000,1.000,1.000,84
periodo,1.000,1.000,1.000,1.000,480
prazo_dias,1.000,1.000,1.000,1.000,98



GLOBAL  precisão=0.996  revocação=0.999  F1=0.998  (7000 entidades em 5353 frases)
Erros restantes: 25 falsos positivos, 5 falsos negativos


## 4. Limitações

**O léxico é fechado.** Projeto fora de `lexico_dominio.json` só é reconhecido se tiver forma de código (`PRJ-2031`). Quando a base real do portfólio estiver disponível, o arquivo passa a ser gerado a partir dela — o código não muda.

**Erro de digitação dentro do termo não é recuperado.** O conjunto contém casos como *"pralisado"* por *"paralisado"*; o casamento é exato após normalizar caixa e acento. Casamento aproximado é possível, mas troca revocação por falso positivo e deve ser decidido com dados reais.

**Entidades que compartilham palavras competem entre si.** Em *"quem assinou o termo de abertura do projeto de sinalização CBTC?"*, o documento `Termo de Abertura do Projeto` e o projeto `projeto de sinalização CBTC` disputam a palavra "projeto"; o trecho mais longo vence e o outro se perde. Afeta 3 das 5.353 frases.

**A ambiguidade é reportada, não resolvida.** Cabe a quem consome — já de posse da intenção — escolher entre os tipos candidatos.

**Não há normalização semântica de período.** `no último trimestre` é devolvido como texto canônico, não como intervalo de datas. A conversão depende da data de referência da consulta e entra com a integração ao banco.